In [0]:
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope ="retailpulse-scope", key= "retailpulse-account-key")
STORAGE_ACCOUNT_NAME = "retailpulsestorage" 
CONTAINER_NAME       = "raw-data"


spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.blob.core.windows.net",
    STORAGE_ACCOUNT_KEY
)



In [0]:
mount_path = dbutils.fs.ls(f"wasbs://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.blob.core.windows.net")

display(mount_path)

In [0]:
file_path = f"{mount_path[0].path}"
df_raw = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)


print(f"Total rows: {df_raw.count():,}")


In [0]:
#Remove duplicate rows

df_deduped = df_raw.dropDuplicates()

dropped = df_raw.count() - df_deduped.count()
print(f"Rows dropped (duplicates): {dropped:,}")
print(f"Rows remaining: {df_deduped.count():,}")

In [0]:
 #Isolate cancelled orders (Invoice starts with 'C')


from pyspark.sql.functions import col, when, lit, to_timestamp

df_cancelled = df_deduped.filter(col("Invoice").startswith("C"))
df_active    = df_deduped.filter(~col("Invoice").startswith("C"))

print(f"Cancelled orders isolated: {df_cancelled.count():,}")
print(f"Active transactions:       {df_active.count():,}")

In [0]:
#Isolate negative quantity rows (returns/adjustments)


df_returns = df_active.filter(col("Quantity") <= 0)
df_active  = df_active.filter(col("Quantity") > 0)

print(f"Return/adjustment rows isolated: {df_returns.count():,}")
print(f"Active transactions remaining:   {df_active.count():,}")

In [0]:
#Remove invalid prices


df_active = df_active.filter(col("Price") > 0)

print(f"Rows after removing invalid prices: {df_active.count():,}")

In [0]:
#Assign GUEST to null Customer IDs


from pyspark.sql.functions import col, when

df_active = df_active.withColumn(
    "Customer ID",
    when(col("`Customer ID`").isNull(), lit("GUEST"))
    .otherwise(col("`Customer ID`"))
)

# Verify no nulls remain
remaining_nulls = df_active.filter(col("`Customer ID`").isNull()).count()
print(f"Remaining null Customer IDs: {remaining_nulls}")

In [0]:
#Fix data types and add total_amount

from pyspark.sql.functions import to_timestamp, round as spark_round

df_clean = df_active \
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")) \
    .withColumn("Customer ID", col("Customer ID").cast("string")) \
    .withColumn("Quantity", col("Quantity").cast("integer")) \
    .withColumn("Price", col("Price").cast("double")) \
    .withColumn("total_amount", spark_round(col("Quantity") * col("Price"), 2))

print("Schema after cleaning:")
df_clean.printSchema()

print("\nSample cleaned rows:")
display(df_clean.limit(10))

In [0]:
#Silver layer quality check


total_clean     = df_clean.count()
null_customers  = df_clean.filter(col("Customer ID").isNull()).count()
neg_qty         = df_clean.filter(col("Quantity") <= 0).count()
bad_price       = df_clean.filter(col("Price") <= 0).count()
null_dates      = df_clean.filter(col("InvoiceDate").isNull()).count()

print("-" * 50)
print("SILVER LAYER — VALIDATION REPORT")
print("-" * 50)
print(f"Clean rows:            {total_clean:,}")
print(f"Null Customer IDs:     {null_customers} ")
print(f"Negative quantities:   {neg_qty}  ")
print(f"Invalid prices:        {bad_price}  ")
print(f"Null dates:            {null_dates}  ")
print("-" * 50)

In [0]:
#Write Silver layer to Blob Storage

base_path      = f"wasbs://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.blob.core.windows.net"
silver_path    = f"{base_path}/silver/clean_transactions"
cancelled_path = f"{base_path}/silver/cancelled_orders"
returns_path   = f"{base_path}/silver/return_adjustments"

df_clean.write.mode("overwrite").parquet(silver_path)
df_cancelled.write.mode("overwrite").parquet(cancelled_path)
df_returns.write.mode("overwrite").parquet(returns_path)

print(" Silver layer written to Blob:")
print(f"   Clean transactions - {silver_path}")
print(f"   Cancelled orders   - {cancelled_path}")
print(f"   Return adjustments - {returns_path}")